In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys


project_path = os.path.join(os.getcwd(),"..","..")
sys.path.append(project_path)
from utils.transformations import reusable


## Dimuser

### AUTOLOADER

In [0]:
# Read Bronze data using Auto Loader with schema evolution enabled

df_user = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format","parquet") \
    .option("cloudFiles.schemaLocation","abfss://silver@deepakazureproject.dfs.core.windows.net/DimUser/schema") \
    .option("schemaEvolutionMode","addNewColumns") \
    .load("abfss://bronze@deepakazureproject.dfs.core.windows.net/DimUser")

In [0]:
# Convert user names to uppercase for standardization

df_user = df_user.withColumn("user_name",upper(col("user_name")))

In [0]:
# Initialize reusable utility class

df_user_obj = reusable()

In [0]:
# Remove Auto Loader rescued data column

df_user = df_user_obj.dropColumn(df_user, ["_rescued_data"])

In [0]:
# Write Silver data using Delta format with checkpointing and trigger-once processing

df_user.writeStream.format("delta"). \
    option("outputMode","append") . \
  option("checkpointLocation","abfss://silver@deepakazureproject.dfs.core.windows.net/DimUser/checkpoint"). \
  trigger(once=True) .\
  option("path","abfss://silver@deepakazureproject.dfs.core.windows.net/DimUser/data") \
  .toTable("spotify_cata.silver.dimuser")

## DimArtist

In [0]:
# Read Bronze data using Auto Loader with schema evolution enabled

df_artist = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format","parquet") \
    .option("cloudFiles.schemaLocation","abfss://silver@deepakazureproject.dfs.core.windows.net/DimArtist/schema") \
    .option("schemaEvolutionMode","addNewColumns") \
    .load("abfss://bronze@deepakazureproject.dfs.core.windows.net/DimArtist")

In [0]:
# Initialize reusable utility class

df_artist_obj = reusable()

In [0]:
# Remove Auto Loader rescued data column

df_artist = df_artist_obj.dropColumn(df_artist, ["_rescued_data"])

In [0]:
# Write Silver data using Delta format with checkpointing and trigger-once processing

df_artist.writeStream.format("delta"). \
 option("outputMode","append") . \
  option("checkpointLocation","abfss://silver@deepakazureproject.dfs.core.windows.net/DimArtist/checkpoint1"). \
  trigger(once=True) .\
  option("path","abfss://silver@deepakazureproject.dfs.core.windows.net/DimArtist/data") \
  .toTable("spotify_cata.silver.dimartist")

### DimTrack

In [0]:
# Read Bronze data using Auto Loader with schema evolution enabled

df_track = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format","parquet") \
    .option("cloudFiles.schemaLocation","abfss://silver@deepakazureproject.dfs.core.windows.net/DimTrack/schema") \
    .option("schemaEvolutionMode","addNewColumns") \
    .load("abfss://bronze@deepakazureproject.dfs.core.windows.net/DimTrack")

In [0]:
# Create duration category based on track duration

df_track = df_track.withColumn(
    "durationFlag",
    when(col("duration_sec") < 150, "low")
    .when(col("duration_sec") < 300, "medium")
    .otherwise("high")
)


In [0]:
# Standardize track names by replacing hyphens with spaces

df_track = df_track.withColumn(
    "track_name",
    regexp_replace(col("track_name"), "-", " ")
)

In [0]:
# Initialize reusable utility class

df_track_obj = reusable()

In [0]:
# Remove Auto Loader rescued data column

df_track = df_track_obj.dropColumn(df_track, ["_rescued_data"])

In [0]:
# Write Silver data using Delta format with checkpointing and trigger-once processing

df_track.writeStream.format("delta"). \
 option("outputMode","append") . \
  option("checkpointLocation","abfss://silver@deepakazureproject.dfs.core.windows.net/DimTrack/checkpoint"). \
  trigger(once=True) .\
  option("path","abfss://silver@deepakazureproject.dfs.core.windows.net/DimTrack/data") \
  .toTable("spotify_cata.silver.dimtrack")

### DimDate

In [0]:
# Read Bronze data using Auto Loader with schema evolution enabled

df_date = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format","parquet") \
    .option("cloudFiles.schemaLocation","abfss://silver@deepakazureproject.dfs.core.windows.net/DimDate/schema") \
    .option("schemaEvolutionMode","addNewColumns") \
    .load("abfss://bronze@deepakazureproject.dfs.core.windows.net/DimDate")

In [0]:
# Initialize reusable utility class

df_date_obj = reusable()

In [0]:
# Remove Auto Loader rescued data column

df_date = df_date_obj.dropColumn(df_date, ["_rescued_data"])

In [0]:
# Write Silver data using Delta format with checkpointing and trigger-once processing

df_date.writeStream.format("delta"). \
 option("outputMode","append") . \
  option("checkpointLocation","abfss://silver@deepakazureproject.dfs.core.windows.net/DimDate/checkpoint"). \
  trigger(once=True) .\
  option("path","abfss://silver@deepakazureproject.dfs.core.windows.net/DimDate/data") \
  .toTable("spotify_cata.silver.dimdate")

## FactStream

In [0]:
# Read Bronze data using Auto Loader with schema evolution enabled

df_stream = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format","parquet") \
    .option("cloudFiles.schemaLocation","abfss://silver@deepakazureproject.dfs.core.windows.net/FactStream/schema") \
    .option("schemaEvolutionMode","addNewColumns") \
    .load("abfss://bronze@deepakazureproject.dfs.core.windows.net/FactStream")

In [0]:
# Remove Auto Loader rescued data column

df_stream = reusable().dropColumn(df_stream,['_rescued_data'])

In [0]:
# Write Silver data using Delta format with checkpointing and trigger-once processing

df_stream.writeStream.format("delta"). \
 option("outputMode","append") . \
  option("checkpointLocation","abfss://silver@deepakazureproject.dfs.core.windows.net/FactStream/checkpoint"). \
  trigger(once=True) .\
  option("path","abfss://silver@deepakazureproject.dfs.core.windows.net/FactStream/data") \
  .toTable("spotify_cata.silver.factstream")

- END